In [ ]:
# use qa_dataset_final.jsonl to create a new dataset with balanced difficulty levels
# create three files: one with easy questions, one with medium questions, and one with hard questions
# in this current path 





import json
import os
import random

def create_balanced_datasets(input_filename="../qa_dataset_final.jsonl"):
    # 1. Verify the source file exists in the current path
    if not os.path.exists(input_filename):
        print(f"❌ Error: Could not find '{input_filename}' in the current directory.")
        print("Please ensure the script is running in the same folder as your dataset.")
        return

    # Initialize lists to separate questions
    easy_pool = []
    medium_pool = []
    hard_pool = []

    print(f"Parsing '{input_filename}'...")
    
    # 2. Read and categorize all entries
    with open(input_filename, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                difficulty = str(data.get("difficulty", "")).lower().strip()
                
                if difficulty == "easy":
                    easy_pool.append(data)
                elif difficulty == "medium":
                    medium_pool.append(data)
                elif difficulty == "hard":
                    hard_pool.append(data)
                else:
                    print(f"Line {line_num}: Unknown difficulty tier '{difficulty}'. Skipping...")
            except json.JSONDecodeError:
                print(f"Line {line_num}: Invalid JSON format. Skipping...")

    # Print original distribution metrics
    print("\n--- Original Dataset Distribution ---")
    print(f"🟢 Easy Questions:   {len(easy_pool)}")
    print(f"🟡 Medium Questions: {len(medium_pool)}")
    print(f"🔴 Hard Questions:   {len(hard_pool)}")

    # 3. Determine the balancing threshold (the size of the smallest group)
    if len(easy_pool) == 0 or len(medium_pool) == 0 or len(hard_pool) == 0:
        print("\n🛑 Error: One or more difficulty categories contain 0 items. Cannot balance dataset.")
        return

    min_count = min(len(easy_pool), len(medium_pool), len(hard_pool))
    print(f"\n⚖️ Balancing Target: Extracting exactly {min_count} random samples per difficulty level.")

    # 4. Shuffle pools for a fair, randomized balanced subset
    random.seed(42)  # Fixed seed for deterministic, repeatable runs
    random.shuffle(easy_pool)
    random.shuffle(medium_pool)
    random.shuffle(hard_pool)

    # Slice the pools to match the minimum count
    balanced_datasets = {
        "easy_questions.jsonl": easy_pool[:min_count],
        "medium_questions.jsonl": medium_pool[:min_count],
        "hard_questions.jsonl": hard_pool[:min_count]
    }

    # 5. Write out the 3 separate balanced files in the current directory
    print("\nWriting balanced outputs to current path...")
    for filename, items in balanced_datasets.items():
        with open(filename, 'w', encoding='utf-8') as out_file:
            for item in items:
                # ensure_ascii=False preserves original unicode characters from your PDFs
                out_file.write(json.dumps(item, ensure_ascii=False) + "\n")
        print(f"✅ Generated '{filename}' ({len(items)} rows)")

    print("\n🎉 Dataset successfully balanced and split!")

if __name__ == "__main__":
    create_balanced_datasets()



Parsing '../qa_dataset_final.jsonl'...

--- Original Dataset Distribution ---
🟢 Easy Questions:   16759
🟡 Medium Questions: 5211
🔴 Hard Questions:   173

⚖️ Balancing Target: Extracting exactly 173 random samples per difficulty level.

Writing balanced outputs to current path...
✅ Generated 'easy_questions.jsonl' (173 rows)
✅ Generated 'medium_questions.jsonl' (173 rows)
✅ Generated 'hard_questions.jsonl' (173 rows)

🎉 Dataset successfully balanced and split!


For now just doing 173 from each once experiment is successull, will do the full question list generated from PDFs. 

In [ ]:
# answer -> ground_truth_answer 
# evidence -> gold_context

In [3]:
# baseline rag 

%pip install langchain-ollama

Note: you may need to restart the kernel to use updated packages.


In [6]:
import os

print("Current Working Directory (Where your notebook is running):")
print(os.getcwd())

print("\nFolders visible in the current directory:")
try:
    print(os.listdir('../docs'))
except Exception as e:
    print(f"Error listing directory: {e}")

print("\nFolders visible in the parent directory (..):")
try:
    print(os.listdir('..'))
except Exception as e:
    print(f"Error listing parent directory: {e}")

Current Working Directory (Where your notebook is running):
/Users/sterinsaji/Desktop/rag-research/notebooks/files-distributed-based-on-difficulty

Folders visible in the current directory:
['.DS_Store', 'NIST.SP.800-171r3.pdf.jsonl', 'rfc9110.pdf', 'NIST.CSWP.29.pdf', 'wellarchitected-framework.pdf.jsonl', 'OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf.jsonl', 'NIST.SP.800-53r5.pdf.jsonl', 'NIST.CSWP.29.pdf.jsonl', 'NIST.SP.800-53r5.pdf', 'rfc9110.pdf.jsonl', 'NIST.SP.800-171r3.pdf', 'OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf', 'wellarchitected-framework.pdf']

Folders visible in the parent directory (..):
['experiment_config.json', 'raw_llm_outputs.jsonl', 'main.ipynb', 'qa_dataset_7513_hf.jsonl', 'qa_checkpoint.jsonl', '.DS_Store', 'chunking.ipynb', 'chunks_ollama.jsonl', 'files-distributed-based-on-difficulty', 'qa_dataset.jsonl', 'chunks.jsonl', 'qa_dataset_for_rag.jsonl', 'docs', 'qa_dataset_final_copy.jsonl', 'experiment_config_ollama.

In [7]:
import os
import sys
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# ============================================================
# CONFIGURATION
# ============================================================

# DOUBLE-CHECK: Ensure this folder matches your actual directory ('doc' vs 'docs')
PDF_PATHS = [
    "../docs/NIST.CSWP.29.pdf",
    "../docs/NIST.SP.800-53r5.pdf",
    "../docs/NIST.SP.800-171r3.pdf",
    "../docs/OWASP_Application_Security_Verification_Standard_5.0.0_en (1).pdf",
    "../docs/rfc9110.pdf",
    "../docs/wellarchitected-framework.pdf"
]

OLLAMA_MODEL = "qwen2.5:7b"
INDEX_PATH = "faiss_index"

# ============================================================
# STEP 1 — INITIALIZE EMBEDDING MODEL
# ============================================================
print("Loading embedding model...\n")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# ============================================================
# STEP 2 — LOAD OR CREATE FAISS VECTOR DATABASE
# ============================================================
if os.path.exists(INDEX_PATH):
    print("Found existing FAISS index locally. Loading index...\n")
    vectorstore = FAISS.load_local(INDEX_PATH, embeddings, allow_dangerous_deserialization=True)
    print("FAISS index loaded from disk successfully.\n")

else:
    print("No local index found. Parsing PDFs...\n")
    documents = []
    files_found = 0
    
    for path in tqdm(PDF_PATHS, desc="Processing PDFs"):
        if os.path.exists(path):
            files_found += 1
            loader = PyPDFLoader(path)
            for page in loader.lazy_load():
                documents.append(page)
        else:
            print(f"\n❌ Error: File NOT found at: {os.path.abspath(path)}")

    # Guardrail 1: Check if any files were located
    if files_found == 0:
        print("\n🛑 CRITICAL ERROR: None of the PDF files could be found. Please check your 'PDF_PATHS' configurations.")
        sys.exit(1)

    print(f"\nLoaded {len(documents)} total pages across all files.\n")

    # Guardrail 2: Check if pages contain actual text
    total_text_length = sum(len(doc.page_content.strip()) for doc in documents)
    if total_text_length == 0:
        print("\n🛑 CRITICAL ERROR: Loaded pages contain 0 characters of text. Your PDFs might be scanned images lacking an OCR layer.")
        sys.exit(1)

    print("Splitting documents into chunks...\n")
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000, 
        chunk_overlap=200
    )
    all_chunks = splitter.split_documents(documents)
    print(f"Created {len(all_chunks)} chunks.\n")

    # Guardrail 3: Final check before feeding to FAISS
    if not all_chunks:
        print("\n🛑 CRITICAL ERROR: Chunking resulted in 0 text blocks. Cannot build FAISS index.")
        sys.exit(1)

    print("Creating FAISS vector database...")
    vectorstore = FAISS.from_documents(all_chunks, embeddings)
    print("FAISS index created successfully.\n")
    
    vectorstore.save_local(INDEX_PATH)
    print(f"FAISS index saved locally to '{INDEX_PATH}' for future use.\n")

# ============================================================
# STEP 3 — CREATE RETRIEVER
# ============================================================
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)
print("Retriever created.\n")

# ============================================================
# STEP 4 — CONNECT TO LOCAL OLLAMA MODEL
# ============================================================
print(f"Connecting to local Ollama ({OLLAMA_MODEL})...\n")
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0
)
print("Ollama LLM connected.\n")

# ============================================================
# STEP 5 — BUILD RAG CHAIN & INVOKE
# ============================================================
prompt = ChatPromptTemplate.from_template("""
You are an expert cybersecurity and architecture assistant. 
Answer the question comprehensively using ONLY the provided context.

If the answer cannot be confidently derived from the context, say:
"I could not find the answer in the provided documents."

<context>
{context}
</context>

Question: {input}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)



/Users/sterinsaji/miniconda3/envs/rag_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...



/var/folders/04/ycg4c8hs6tncxvm8l19zwkj80000gn/T/ipykernel_21035/851683318.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5618.00it/s]


No local index found. Parsing PDFs...



Processing PDFs: 100%|██████████| 6/6 [01:05<00:00, 10.85s/it]



Loaded 1960 total pages across all files.

Splitting documents into chunks...

Created 6332 chunks.

Creating FAISS vector database...
FAISS index created successfully.

FAISS index saved locally to 'faiss_index' for future use.

Retriever created.

Connecting to local Ollama (qwen2.5:7b)...

Ollama LLM connected.



In [8]:
print("Running Query...\n")
response = rag_chain.invoke(
    "What are two functions that accountability and traceability serve?"
)

print("--- RESPONSE ---")
print(response)

Running Query...

--- RESPONSE ---
Accountability and traceability serve two primary functions:

1. They enable tracing security-relevant actions to the entity on whose behalf the action is being taken.
2. They provide non-repudiation by recording details about actions that affect system security, making it difficult or impossible to change the audit trail once an action is recorded.


In [12]:
import os
import json
from tqdm import tqdm

def generate_experimental_dataset(input_files, output_file="experiment_dataset.jsonl"):
    """
    Processes JSONL files through the RAG chain, tracks progress via tqdm, 
    streams entries instantly to disk, and resumes automatically if restarted.
    """
    processed_ids = set()
    
    # ============================================================
    # STEP 1 — CHECK FOR EXISTING PROGRESS (RESUME CAPABILITY)
    # ============================================================
    if os.path.exists(output_file):
        print(f"🔄 Found existing output file '{output_file}'. Loading progress...")
        try:
            with open(output_file, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        existing_data = json.loads(line)
                        existing_id = existing_data.get("id")
                        if existing_id:
                            processed_ids.add(existing_id)
                    except json.JSONDecodeError:
                        continue # Skip corrupted rows inside the output file
            print(f"⏮️ Found {len(processed_ids)} items already processed. They will be skipped automatically.\n")
        except Exception as e:
            print(f"⚠️ Could not read existing output file smoothly ({e}). Starting fresh.")
    else:
        print(f"🚀 Starting fresh Experimental Dataset Generation...")
        print(f"📦 Output will stream directly to: '{output_file}'\n")

    new_items_processed = 0

    # ============================================================
    # STEP 2 — OPEN OUTPUT FILE IN APPEND MODE ('a')
    # ============================================================
    with open(output_file, 'a', encoding='utf-8') as outfile:
        
        for input_file in input_files:
            if not os.path.exists(input_file):
                print(f"❌ Error: Could not find '{input_file}'. Skipping...")
                continue

            # Compute lines ahead of time for precise percentage calculation
            with open(input_file, 'r', encoding='utf-8') as f:
                total_lines = sum(1 for line in f if line.strip())

            with open(input_file, 'r', encoding='utf-8') as infile:
                progress_bar = tqdm(
                    infile, 
                    total=total_lines, 
                    desc=f"📄 Processing {input_file:<22}", 
                    unit="query"
                )

                for line_num, line in enumerate(progress_bar, 1):
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        data = json.loads(line)
                        item_id = data.get("id")
                        question = data.get("question", "")

                        # ------------------------------------------------
                        # CHECKPOINT TRIGGER: Skip if completed previously
                        # ------------------------------------------------
                        if item_id in processed_ids:
                            progress_bar.set_postfix_str(f"Skipping ID: {item_id[:6]}... Done")
                            continue

                        # Update UI with current active target
                        progress_bar.set_postfix_str(f"ID: {item_id[:6]}... | Q: '{question[:25]}...'")

                        # Run query through local Ollama instance
                        rag_answer = rag_chain.invoke(question)

                        # Formulate data record
                        experimental_item = {
                            "id": item_id,
                            "question": question,
                            "answer": data.get("answer"),
                            "evidence": data.get("evidence"),
                            "question_type": data.get("question_type"),
                            "difficulty": data.get("difficulty"),
                            "chunk_id": data.get("chunk_id"),
                            "source_doc": data.get("source_doc"),
                            "rag_answer": rag_answer
                        }

                        # ------------------------------------------------
                        # STREAM & FLUSH: Commit immediately to disk
                        # ------------------------------------------------
                        outfile.write(json.dumps(experimental_item, ensure_ascii=False) + "\n")
                        outfile.flush()  # Forces hardware synchronization immediately
                        
                        new_items_processed += 1

                    except json.JSONDecodeError:
                        print(f"\n⚠️ Line {line_num} in '{input_file}': Invalid JSON format. Skipping...")
                    except Exception as e:
                        print(f"\n❌ Error processing item {data.get('id', 'Unknown')} due to: {e}. Skipping...")

    print(f"\n🎉 Process finished! Dataset at '{output_file}'. Added {new_items_processed} new entries this run.")

# ==========================================
# EXECUTION
# ==========================================
generate_experimental_dataset(
    input_files=["easy_questions.jsonl", "medium_questions.jsonl", "hard_questions.jsonl"], 
    output_file="experiment_dataset.jsonl"
)

🚀 Starting fresh Experimental Dataset Generation...
📦 Output will stream directly to: 'experiment_dataset.jsonl'



📄 Processing hard_questions.jsonl  : 100%|██████████| 173/173 [29:01<00:00, 10.07s/query, ID: c534ea... | Q: 'What are the steps to set...']


🎉 Process finished! Dataset at 'experiment_dataset.jsonl'. Added 519 new entries this run.


In [13]:
import os
import json
import re
from tqdm import tqdm
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# ============================================================
# CONFIGURATION
# ============================================================
INPUT_FILE = "experiment_dataset.jsonl"
OUTPUT_FILE = "hallucination_dataset.jsonl"
OLLAMA_MODEL = "qwen2.5:7b"

# Initialize the Judge LLM (Temperature 0 for maximum strictness/consistency)
llm = ChatOllama(
    model=OLLAMA_MODEL,
    temperature=0
)

# Define the strict Evaluation Prompt
evaluator_prompt = ChatPromptTemplate.from_template("""
You are an expert evaluator for a Retrieval-Augmented Generation (RAG) system.
Your task is to compare the "Generated Answer" against the provided "Evidence" to detect hallucinations.

A "hallucination" is defined as ANY of the following:
1. The generated answer states facts, numbers, or entities not present in the evidence.
2. The generated answer contradicts the evidence.
3. The generated answer claims the evidence lacks the information, but the evidence actually contains it.

<evidence>
{evidence}
</evidence>

<generated_answer>
{rag_answer}
</generated_answer>

Does the generated answer contain a hallucination based strictly on the provided evidence?
Respond ONLY with a single digit:
1 (if hallucination is detected)
0 (if the answer is fully faithful to the evidence or accurately states the evidence lacks the info)
""")

# Build the evaluation chain
eval_chain = evaluator_prompt | llm

def extract_binary_score(response_text):
    """Safely extracts 1 or 0 from the LLM response string."""
    text = response_text.content.strip()
    # Search for the first occurrence of 0 or 1
    match = re.search(r'[01]', text)
    if match:
        return int(match.group())
    else:
        # Fallback if the LLM completely fails to follow formatting instructions
        return -1 

def evaluate_hallucinations():
    processed_ids = set()
    
    # 1. Check for existing progress (Resume capability)
    if os.path.exists(OUTPUT_FILE):
        print(f"🔄 Found existing output file '{OUTPUT_FILE}'. Loading progress...")
        try:
            with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        existing_data = json.loads(line)
                        existing_id = existing_data.get("id")
                        if existing_id:
                            processed_ids.add(existing_id)
                    except json.JSONDecodeError:
                        continue 
            print(f"⏮️ Skipping {len(processed_ids)} items already evaluated.\n")
        except Exception as e:
            print(f"⚠️ Could not read existing output file ({e}). Starting fresh.")
    else:
        print(f"🚀 Starting Hallucination Evaluation...")

    if not os.path.exists(INPUT_FILE):
        print(f"❌ Error: Input file '{INPUT_FILE}' not found. Please run the generation script first.")
        return

    # Count total lines for tqdm
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        total_lines = sum(1 for line in f if line.strip())

    items_evaluated = 0

    # 2. Open output file in append mode and stream evaluations
    with open(INPUT_FILE, 'r', encoding='utf-8') as infile, \
         open(OUTPUT_FILE, 'a', encoding='utf-8') as outfile:
        
        progress_bar = tqdm(
            infile, 
            total=total_lines, 
            desc=f"🕵️ Evaluating Answers", 
            unit="item"
        )

        for line_num, line in enumerate(progress_bar, 1):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
                item_id = data.get("id")

                # Skip if already evaluated
                if item_id in processed_ids:
                    progress_bar.set_postfix_str(f"Skipping ID: {item_id[:6]}")
                    continue

                evidence = data.get("evidence", "")
                rag_answer = data.get("rag_answer", "")

                # Set UI update
                progress_bar.set_postfix_str(f"Evaluating ID: {item_id[:6]}")

                # If there's no rag_answer or no evidence, we can flag it immediately or evaluate
                if not rag_answer:
                    hallucination_score = 1 # Empty response when expected counts as failure
                else:
                    # Invoke the LLM Judge
                    eval_response = eval_chain.invoke({
                        "evidence": evidence,
                        "rag_answer": rag_answer
                    })
                    hallucination_score = extract_binary_score(eval_response)

                # Attach the score to the data object
                data["hallucination"] = hallucination_score

                # Write to disk immediately
                outfile.write(json.dumps(data, ensure_ascii=False) + "\n")
                outfile.flush()
                
                items_evaluated += 1

            except json.JSONDecodeError:
                print(f"\n⚠️ Line {line_num}: Invalid JSON format. Skipping...")
            except Exception as e:
                print(f"\n❌ Error evaluating item {data.get('id', 'Unknown')}: {e}. Skipping...")

    print(f"\n🎉 Evaluation Complete! Generated '{OUTPUT_FILE}'. Evaluated {items_evaluated} new entries.")

if __name__ == "__main__":
    evaluate_hallucinations()

🚀 Starting Hallucination Evaluation...


🕵️ Evaluating Answers: 100%|██████████| 519/519 [10:03<00:00,  1.16s/item, Evaluating ID: c534ea]


🎉 Evaluation Complete! Generated 'hallucination_dataset.jsonl'. Evaluated 519 new entries.


In [16]:
# calculate the number of hallucinations in the hallucination_dataset.jsonl file
def calculate_hallucination_stats(input_file="hallucination_dataset.jsonl"):
    if not os.path.exists(input_file):
        print(f"❌ Error: Input file '{input_file}' not found. Please run the hallucination evaluation script first.")
        return

    total_items = 0
    hallucination_count = 0

    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data = json.loads(line)
                total_items += 1
                if data.get("hallucination") == 1:
                    hallucination_count += 1
            except json.JSONDecodeError:
                print(f"⚠️ Invalid JSON format in line: {line}. Skipping...")

    if total_items == 0:
        print("🛑 No valid entries found in the dataset.")
        return

    hallucination_percentage = (hallucination_count / total_items) * 100

    print("\n--- Hallucination Statistics ---")
    print(f"Total Items Evaluated: {total_items}")
    print(f"Hallucinations Detected: {hallucination_count}")
    print(f"Percentage of Hallucinations: {hallucination_percentage:.2f}%")
    
    
calculate_hallucination_stats()


--- Hallucination Statistics ---
Total Items Evaluated: 519
Hallucinations Detected: 101
Percentage of Hallucinations: 19.46%
